# SSD MobileNet v1 Training with Roboflow Dataset

DCC Lab - 2026

[HAM]

This notebook trains an SSD MobileNet v1 detector with Roboflow dataset input.

Adapted from: [Jetson Inference - Re-training SSD-Mobilenet
](https://github.com/dusty-nv/jetson-inference/blob/master/docs/pytorch-ssd.md)

### Steps
1. Set the runtime as GPU
2. Fill the Roboflow credentials (cell 2)
3. Set the training parameters, mainly batch size and epoch (cell 2)
4. Run all of the cells
5. Make sure the internet is not lost and regularly check the training notebook, so the training process is not interrupted.
6. When the training finished, takes the ".onnx", ".onnx.data" and "labels.txt" file for inference process using jetson nano.

In [ ]:
# use drive, so the training documentation is saved
import os

USE_DRIVE = True  # set False to skip

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/ssd_mobilenet_v1_training'  #IMPORTANT: Change This Accordingly!
    print(f'Saving to Google Drive: {OUTPUT_DIR}')
else:
    OUTPUT_DIR = '/content/training'
    print(f'Saving locally (lost on disconnect): {OUTPUT_DIR}')

os.makedirs(OUTPUT_DIR, exist_ok=True)
%cd {OUTPUT_DIR}

In [ ]:
# Clone the dusty-nv pytorch-ssd repo (skip if already cloned)
import os
if not os.path.exists('pytorch-ssd'):
    !git clone https://github.com/dusty-nv/pytorch-ssd.git

# go inside the folders
%cd pytorch-ssd

# Download pretrained MobileNet-v1 SSD backbone weights
if not os.path.exists('models/mobilenet-v1-ssd-mp-0_675.pth'):
    !mkdir -p models
    !wget https://nvidia.box.com/shared/static/djf5w54rjvpqocsiztzaandq1m3avr7c.pth \\
          -O models/mobilenet-v1-ssd-mp-0_675.pth

# Install pytorch-ssd requirements + Roboflow SDK
!pip3 install -v -r requirements.txt
!pip install roboflow -q


Cloning into 'pytorch-ssd'...
remote: Enumerating objects: 1007, done.
remote: Total 1007 (delta 0), reused 0 (delta 0), pack-reused 1007 (from 1)
Receiving objects: 100% (1007/1007), 1.11 MiB | 3.62 MiB/s, done.
Resolving deltas: 100% (665/665), done.
/content/pytorch-ssd
--2026-05-16 14:17:38--  https://nvidia.box.com/shared/static/djf5w54rjvpqocsiztzaandq1m3avr7c.pth
Resolving nvidia.box.com (nvidia.box.com)... 74.112.186.157, 2620:117:bff0:12d::
Connecting to nvidia.box.com (nvidia.box.com)|74.112.186.157|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: /public/static/djf5w54rjvpqocsiztzaandq1m3avr7c.pth [following]
--2026-05-16 14:17:38--  https://nvidia.box.com/public/static/djf5w54rjvpqocsiztzaandq1m3avr7c.pth
Reusing existing connection to nvidia.box.com:443.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://nvidia.app.box.com/public/static/djf5w54rjvpqocsiztzaandq1m3avr7c.pth [following]
--2026-05-16 14:17

In [ ]:
# ════════════════════════════════════════════════════
# CONFIGURE YOUR ROBOFLOW DATASET HERE
# ════════════════════════════════════════════════════

# get this data from your dataset/versions --> export --> download, with link options

ROBOFLOW_API_KEY   = 'API-KEY'       # <-- paste your Roboflow API key
ROBOFLOW_WORKSPACE = 'workspace-name'            # <-- e.g. 'my-team'
ROBOFLOW_PROJECT   = 'project-name'       # <-- e.g. 'smart-trash-detector'
ROBOFLOW_VERSION   = 1                            # <-- dataset version (integer)

DATASET_NAME = ROBOFLOW_PROJECT           # used as the folder name under data/
MODEL_DIR    = f'models/{DATASET_NAME}'  # checkpoint output directory
BATCH_SIZE   = 4
EPOCHS       = 30

print(f'Dataset : {ROBOFLOW_WORKSPACE}/{ROBOFLOW_PROJECT} v{ROBOFLOW_VERSION}')
print(f'Model dir: {MODEL_DIR}')


In [ ]:
from roboflow import Roboflow

# Data preprocessing

rf      = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
version = project.version(ROBOFLOW_VERSION)

# Download in Pascal VOC format — converted to Open Images CSV in the next cell
dataset = version.download('voc', location=f'data/{DATASET_NAME}_raw', overwrite=True)

print('Download complete:', dataset.location)
!find data/{DATASET_NAME}_raw -maxdepth 2 | head -40


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to data/smart-trash-detector_raw in voc:: 100%|██████████| 1640/1640 [00:00<00:00, 2872.43it/s]


Download complete: /content/pytorch-ssd/data/smart-trash-detector_raw
data/smart-trash-detector_raw
data/smart-trash-detector_raw/train
data/smart-trash-detector_raw/train/IMG_20260426_150415_jpg.rf.31faa767814bb25abed6d607f1e7a85b.jpg
data/smart-trash-detector_raw/train/IMG_20260426_124752_jpg.rf.ef7627ba87448e0cfd57494169de5756.xml
data/smart-trash-detector_raw/train/IMG_20260505_155644_HEIC.rf.43944ef83fef65bfcdffea108551a67e.xml
data/smart-trash-detector_raw/train/IMG_20260505_160227_HEIC.rf.6c25916208d7641eed90e46dc9b6d3f6.xml
data/smart-trash-detector_raw/train/IMG_20260505_151316_jpg.rf.9ad6e335150446ed7779a729b29e5e6b.jpg
data/smart-trash-detector_raw/train/IMG_20260505_153234_HEIC.rf.967c9c3a22befcc773a7a134e9733806.xml
data/smart-trash-detector_raw/train/IMG_20260426_134248_jpg.rf.f0cf0255f5239db7e4a262ead1d12a50.xml
data/smart-trash-detector_raw/train/IMG_20260425_151554_jpg.rf.3c58e3ca89a81281a40be452d4843d17.jpg
data/smart-trash-detector_raw/train/IMG_20260425_161004_jpg.r

In [ ]:
"""
Preprocessing 2:

Convert a Roboflow Pascal-VOC export to the Open Images CSV format
expected by pytorch-ssd.

Open Images CSV columns used by pytorch-ssd:
    ImageID, Source, ClassName, Confidence,
    XMin, XMax, YMin, YMax,
    IsOccluded, IsTruncated, IsGroupOf, IsDepiction, IsInside

Coordinates are normalised to [0, 1].
"""

import os, glob, csv, shutil
from pathlib import Path
import xml.etree.ElementTree as ET

RAW_DIR = Path(f'data/{DATASET_NAME}_raw')
OUT_DIR = Path(f'data/{DATASET_NAME}')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Roboflow VOC exports may use: train/, valid/, val/, test/
SPLIT_MAP = {'train': 'train', 'valid': 'validation', 'val': 'validation', 'test': 'test'}

all_labels = set()

def convert_split(src_dir: Path, split_name: str):
    img_out = OUT_DIR / split_name
    img_out.mkdir(parents=True, exist_ok=True)

    xml_files = list(src_dir.glob('*.xml'))
    if not xml_files:
        print(f'  [skip] no XML files in {src_dir}')
        return []

    rows = []
    for xml_path in xml_files:
        tree  = ET.parse(xml_path)
        root  = tree.getroot()

        xml_filename = root.findtext('filename') or ''
        size_node = root.find('size')
        img_w = int(size_node.findtext('width'))
        img_h = int(size_node.findtext('height'))

        # Roboflow sometimes writes <filename>foo.jpg</filename> even though
        # the image on disk is also foo.jpg, leading to 'foo.jpg.jpg' if we
        # naively append an extension.  Resolve by matching on the stem only.
        xml_stem = Path(xml_filename).stem  # strip extension from the tag value
        found_img = None
        for search_dir in [src_dir, src_dir.parent]:
            for ext in ['.jpg', '.jpeg', '.png', '.bmp', '.JPG', '.JPEG', '.PNG']:
                c = search_dir / (xml_stem + ext)
                if c.exists():
                    found_img = c
                    break
            if found_img:
                break

        if found_img is None:
            print(f'  [warn] image not found for: {xml_path.name}')
            continue

        # Use the real filename on disk (no double-extension risk)
        filename = found_img.name
        shutil.copy2(found_img, img_out / filename)

        for obj in root.findall('object'):
            label = obj.findtext('name')
            all_labels.add(label)
            bb = obj.find('bndbox')
            xmin = max(0.0, float(bb.findtext('xmin')) / img_w)
            xmax = min(1.0, float(bb.findtext('xmax')) / img_w)
            ymin = max(0.0, float(bb.findtext('ymin')) / img_h)
            ymax = min(1.0, float(bb.findtext('ymax')) / img_h)
            rows.append([filename, 'freeform', label, 1,
                          xmin, xmax, ymin, ymax, 0, 0, 0, 0, 0])
    return rows

# ── Discover splits ────────────────────────────────────────────────────────────
split_rows = {}
for sub in sorted(RAW_DIR.iterdir()):
    if sub.is_dir() and sub.name.lower() in SPLIT_MAP:
        mapped = SPLIT_MAP[sub.name.lower()]
        print(f'Processing split: {sub.name} -> {mapped}')
        rows = convert_split(sub, mapped)
        if rows:
            # If the same mapped split appears twice (e.g. valid + val), merge
            split_rows.setdefault(mapped, []).extend(rows)

# Flat layout fallback
if not split_rows:
    print('Flat layout detected — treating everything as train')
    rows = convert_split(RAW_DIR, 'train')
    if rows:
        split_rows['train'] = rows

# ── Write CSVs ─────────────────────────────────────────────────────────────────
CSV_HEADER = ['ImageID','Source','ClassName','Confidence',
               'XMin','XMax','YMin','YMax',
               'IsOccluded','IsTruncated','IsGroupOf','IsDepiction','IsInside']

def write_csv(path, rows):
    with open(path, 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(CSV_HEADER)
        w.writerows(rows)
    print(f'  wrote {len(rows):,} rows -> {path}')

if 'train' in split_rows:
    write_csv(OUT_DIR / 'sub-train-annotations-bbox.csv', split_rows['train'])

val_key = 'validation' if 'validation' in split_rows else ('test' if 'test' in split_rows else None)
if val_key:
    write_csv(OUT_DIR / 'sub-test-annotations-bbox.csv', split_rows[val_key])
else:
    print('[warn] No validation/test split found. pytorch-ssd needs sub-test-annotations-bbox.csv')
    print('       Falling back: copying train CSV as validation.')
    write_csv(OUT_DIR / 'sub-test-annotations-bbox.csv', split_rows.get('train', []))

# ── Write labels.txt ──────────────────────────────────────────────────────────
sorted_labels = sorted(all_labels)
(OUT_DIR / 'labels.txt').write_text('\n'.join(sorted_labels) + '\n')

print(f'\nClasses ({len(sorted_labels)}): {sorted_labels}')
print('\nOutput directory:')
!find data/{DATASET_NAME} -maxdepth 2 | head -30


Processing split: test -> test
Processing split: train -> train
Processing split: valid -> validation
  wrote 6,325 rows -> data/smart-trash-detector/sub-train-annotations-bbox.csv
  wrote 375 rows -> data/smart-trash-detector/sub-test-annotations-bbox.csv

Classes (3): ['Sterofoam', 'lain-lainya', 'plastikk']

Output directory:
data/smart-trash-detector
data/smart-trash-detector/validation
data/smart-trash-detector/validation/IMG_20260505_152721_jpg.rf.3bd532a541c97ec435a030d803ad9675.jpg
data/smart-trash-detector/validation/1777199236311_jpg.rf.dd999ccda2427e0d2fb26ae17d572547.jpg
data/smart-trash-detector/validation/1777199236686_jpg.rf.e50e585d7ca360cfba1341b97ee70f09.jpg
data/smart-trash-detector/validation/1777199236748_jpg.rf.899291123beb2ae0cf1a737015c366df.jpg
data/smart-trash-detector/validation/1777199236758_jpg.rf.ba0f056c58fc4e22fb421346f3f93432.jpg
data/smart-trash-detector/validation/IMG_20260505_160533_HEIC.rf.ae4aeb3e146bcea08258d95acde270e1.jpg
data/smart-trash-detect

In [ ]:
# this cell is for extension removal workaround. Somehow, the process will add the extension itself
# going with this workaround is faster than changes the training code everytime (maybe)
import pandas as pd
from pathlib import Path

for csv_file in [
    f'data/{DATASET_NAME}/sub-train-annotations-bbox.csv',
    f'data/{DATASET_NAME}/sub-test-annotations-bbox.csv',
]:
    p = Path(csv_file)
    if not p.exists():
        continue
    df = pd.read_csv(p)
    # Strip the .jpg/.jpeg/.png extension from ImageID — open_images.py adds .jpg itself
    df['ImageID'] = df['ImageID'].apply(lambda x: Path(x).stem)
    df.to_csv(p, index=False)
    print(f'Fixed: {csv_file}')
    print(df['ImageID'].head(5).tolist())

Fixed: data/smart-trash-detector/sub-train-annotations-bbox.csv
['IMG_20260426_124752_jpg.rf.ef7627ba87448e0cfd57494169de5756', 'IMG_20260426_124752_jpg.rf.ef7627ba87448e0cfd57494169de5756', 'IMG_20260426_124752_jpg.rf.ef7627ba87448e0cfd57494169de5756', 'IMG_20260426_124752_jpg.rf.ef7627ba87448e0cfd57494169de5756', 'IMG_20260426_124752_jpg.rf.ef7627ba87448e0cfd57494169de5756']
Fixed: data/smart-trash-detector/sub-test-annotations-bbox.csv
['1777199236311_jpg.rf.dd999ccda2427e0d2fb26ae17d572547', '1777199236311_jpg.rf.dd999ccda2427e0d2fb26ae17d572547', '1777199236311_jpg.rf.dd999ccda2427e0d2fb26ae17d572547', '1777199236311_jpg.rf.dd999ccda2427e0d2fb26ae17d572547', '1777199236311_jpg.rf.dd999ccda2427e0d2fb26ae17d572547']


In [ ]:
# test and training is misinterpreted while training,
# so just combine it for simplicity
import shutil
from pathlib import Path

src = Path(f'data/{DATASET_NAME}/validation')
dst = Path(f'data/{DATASET_NAME}/test')

for f in src.glob('*'):
    shutil.move(str(f), dst / f.name)

src.rmdir()
print(f"Merged validation/ into test/")
print(f"Total images in test/: {len(list(dst.glob('*')))}")

Merged validation/ into test/
Total images in test/: 102


In [ ]:
# The parameters (batch size and epoch) is set on cell 2.
import os
os.makedirs(MODEL_DIR, exist_ok=True)

!python3 train_ssd.py\
    --data=data/{DATASET_NAME}\
    --model-dir={MODEL_DIR} \
    --batch-size={BATCH_SIZE} \
    --epochs={EPOCHS}


2026-05-16 14:18:15.413758: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/content/pytorch-ssd/vision/utils/box_utils.py:88: SyntaxWarning: invalid escape sequence '\_'
  $$predicted\_center * center_variance = \frac {real\_center - prior\_center} {prior\_hw}$$
2026-05-16 14:18:25 - Using CUDA...
2026-05-16 14:18:25 - Namespace(dataset_type='open_images', datasets=['data/smart-trash-detector'], balance_data=False, net='mb1-ssd', resolution=300, freeze_base_net=False, freeze_net=False, mb2_width_mult=1.0, base_net=None, pretrained_ssd='models/mobilenet-v1-ssd-mp-0_675.pth', resume=None, lr=0.01, momentum=0.9, weight_decay=0.0005, gamma=0.1, base_net_lr=0.001, extra_layers_lr=None, scheduler='cosine', milestones='80,100', t_max=100, batch_size=4,

In [ ]:
# install this one first
!pip3 install onnxscript

In [ ]:
# convert
!python3 onnx_export.py --model-dir={MODEL_DIR}


Namespace(net='ssd-mobilenet', input='', output='', labels='labels.txt', resolution=300, batch_size=1, model_dir='models/smart-trash-detector')
=> running on device cuda:0
=> found best checkpoint with loss 4.9741279813978405 (models/smart-trash-detector/mb1-ssd-Epoch-25-Loss-4.9741279813978405.pth)
=> creating network:  ssd-mobilenet
=> num classes:       4
=> resolution:        300x300
=> loading checkpoint:  models/smart-trash-detector/mb1-ssd-Epoch-25-Loss-4.9741279813978405.pth
=> exporting model to ONNX...
W0516 14:28:18.058000 4556 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0516 14:28:18.059000 4556 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampl

In [ ]:
# Copy labels.txt into model dir so detectnet in jetson can find it
import shutil
src_lbl = f'data/{DATASET_NAME}/labels.txt'
dst_lbl = f'{MODEL_DIR}/labels.txt'
if not os.path.exists(dst_lbl):
    shutil.copy(src_lbl, dst_lbl)
print('Model artefacts:')
!ls -lh {MODEL_DIR}


Model artefacts:
total 819M
-rw-r--r-- 1 root root   41 May 16 14:18 labels.txt
-rw-r--r-- 1 root root  27M May 16 14:18 mb1-ssd-Epoch-0-Loss-6.5378648440043134.pth
-rw-r--r-- 1 root root  27M May 16 14:21 mb1-ssd-Epoch-10-Loss-5.330060799916585.pth
-rw-r--r-- 1 root root  27M May 16 14:22 mb1-ssd-Epoch-11-Loss-5.393729633755154.pth
-rw-r--r-- 1 root root  27M May 16 14:22 mb1-ssd-Epoch-12-Loss-5.745226462682088.pth
-rw-r--r-- 1 root root  27M May 16 14:22 mb1-ssd-Epoch-13-Loss-5.246542665693495.pth
-rw-r--r-- 1 root root  27M May 16 14:23 mb1-ssd-Epoch-14-Loss-5.477414555019802.pth
-rw-r--r-- 1 root root  27M May 16 14:23 mb1-ssd-Epoch-15-Loss-6.047782977422078.pth
-rw-r--r-- 1 root root  27M May 16 14:23 mb1-ssd-Epoch-16-Loss-5.3030006885528564.pth
-rw-r--r-- 1 root root  27M May 16 14:23 mb1-ssd-Epoch-17-Loss-5.231682194603814.pth
-rw-r--r-- 1 root root  27M May 16 14:24 mb1-ssd-Epoch-18-Loss-5.0449395179748535.pth
-rw-r--r-- 1 root root  27M May 16 14:24 mb1-ssd-Epoch-19-Loss-5.614

In [ ]:
# ── Inference with detectnet (Jetson / jetson-inference desktop) ──
# Do this directly on Jetson